In [3]:
# In[1]: Import the module
from pathlib import Path
import sys
import pandas as pd
import os

# Resolve src/ robustly whether the kernel's cwd is the project root
# (RAATS_Test/) or the notebooks/ folder itself — VS Code's Jupyter
# kernel can default to either depending on settings.
cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
src_path = str(project_root / "src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f'cwd: {cwd}')
print(f'project_root resolved to: {project_root}')
print(f'src_path added to sys.path: {src_path}')
assert os.path.isdir(src_path), f'src/ not found at {src_path} — check project_root resolution'

from data.market_data import fetch_price_data, add_technical_indicators, save_data

# In[2]: Define parameters
tickers = ['AAPL']
end = pd.Timestamp.now()
start = end - pd.DateOffset(months=6)  # shorter for test
# Note: a 1-month window will not have enough rows for SMA_50/EMA_50 to
# populate at all (needs 50 trading days). That's expected here — this is
# a plumbing test, not a data-sufficiency test. save_data's min_rows check
# will skip the ticker entirely if there's too little data; lower min_rows
# below if you want to force it through for testing purposes.

# In[3]: Fetch data
data_dict = fetch_price_data(tickers, start, end)
print(f'Fetched {len(data_dict)} ticker(s)')
assert len(data_dict) > 0, 'No data fetched — check yfinance version or rate limiting'

# In[4]: Test add_technical_indicators on one ticker
ticker = tickers[0]
df = data_dict[ticker]

# Verify MultiIndex was already flattened inside fetch_price_data
assert not isinstance(df.columns, pd.MultiIndex), 'Columns are still MultiIndex — flatten fix not applied'

df_with_ind = add_technical_indicators(df.copy())
print('Original columns:', df.columns.tolist())
print('With indicators columns:', df_with_ind.columns.tolist())
print('Indicator columns added:', [c for c in df_with_ind.columns if c not in df.columns])

# In[5]: Validate data cleanliness — dtypes and NaN pattern
print('Dtypes:')
print(df_with_ind.dtypes)

non_numeric_cols = df_with_ind.select_dtypes(exclude=['number']).columns.tolist()
assert not non_numeric_cols, f'Non-numeric columns found (indicates unread MultiIndex/header bug): {non_numeric_cols}'

print('\nNaN counts per column:')
print(df_with_ind.isna().sum())
print('\nAll columns numeric — data is clean.')

# In[6]: Test save_data (writes raw, full-indicator, and clean-indicator CSVs)
test_dir = str(project_root / 'test_pipeline_output')
os.makedirs(test_dir, exist_ok=True)

# Lower min_rows so a short 1-month test window still produces output;
# in production runs (6+ months of data) the default min_rows=50 is fine.
summary = save_data(data_dict, test_dir, min_rows=10)

assert len(summary) > 0, 'save_data produced no output — check min_rows / data length'
print(f'\nTest data saved to {test_dir}')
print('Summary:', summary)

# In[7]: Confirm all three expected output files exist and load cleanly
for ticker, total, clean in summary:
    raw_path = os.path.join(test_dir, 'raw', 'prices', f'{ticker}_ohlcv.csv')
    full_path = os.path.join(test_dir, 'processed', 'indicators', f'{ticker}_indicators.csv')
    clean_path = os.path.join(test_dir, 'processed', 'indicators', f'{ticker}_indicators_clean.csv')

    for path in (raw_path, full_path, clean_path):
        assert os.path.exists(path), f'Expected output file missing: {path}'

    reloaded = pd.read_csv(clean_path, index_col=0, parse_dates=True)
    assert reloaded.isna().sum().sum() == 0, f'{ticker} clean file still contains NaNs'
    assert (reloaded.dtypes != 'object').all(), f'{ticker} clean file has non-numeric columns'

    print(f'{ticker}: all 3 files present, clean file has {len(reloaded)} fully-populated rows.')

print('\nPipeline test passed.')

cwd: c:\Users\Zamuxolo\RAATS_Test\notebooks
project_root resolved to: c:\Users\Zamuxolo\RAATS_Test
src_path added to sys.path: c:\Users\Zamuxolo\RAATS_Test\src
Fetching AAPL...
Fetched 1 ticker(s)
Original columns: ['Close', 'High', 'Low', 'Open', 'Volume']
With indicators columns: ['Close', 'High', 'Low', 'Open', 'Volume', 'SMA_10', 'SMA_20', 'SMA_50', 'EMA_10', 'EMA_20', 'EMA_50', 'RSI_14', 'MACD_12_26_9', 'MACDh_12_26_9', 'MACDs_12_26_9']
Indicator columns added: ['SMA_10', 'SMA_20', 'SMA_50', 'EMA_10', 'EMA_20', 'EMA_50', 'RSI_14', 'MACD_12_26_9', 'MACDh_12_26_9', 'MACDs_12_26_9']
Dtypes:
Price
Close            float64
High             float64
Low              float64
Open             float64
Volume             int64
SMA_10           float64
SMA_20           float64
SMA_50           float64
EMA_10           float64
EMA_20           float64
EMA_50           float64
RSI_14           float64
MACD_12_26_9     float64
MACDh_12_26_9    float64
MACDs_12_26_9    float64
dtype: object

NaN 